In [1]:
%reload_ext autoreload
%autoreload 2

In [16]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

In [ ]:
# for NASA SDE IR benchmark subsample (v1)
from datasets import load_dataset

dataset_path = "nasa-impact/nasa-sde-IR-benchmark-sample-v1"
corpus = load_dataset(
    dataset_path,
    data_files="corpus.jsonl",
    split="train",
    # token=hf_token
).to_pandas()
queries = load_dataset(
    dataset_path,
    data_files="queries.jsonl",
    split="train",
    # token=hf_token
).to_pandas()


corpus["n_tokens"] = corpus["text"].apply(lambda x: len(tokenizer.encode(x)))
queries["n_tokens"] = queries["text"].apply(lambda x: len(tokenizer.encode(x)))


print("Corpus token count:", corpus["n_tokens"].sum())
print("Queries token count:", queries["n_tokens"].sum())

total_tokens = corpus["n_tokens"].sum() + queries["n_tokens"].sum()
print("Total tokens:", total_tokens)


Corpus token count: 10590306
Queries token count: 24955
Total tokens: 10615261


In [ ]:
# for NASA SDE IR benchmark subsample (v2)
from datasets import load_dataset

dataset_path = "nasa-impact/nasa-sde-IR-benchmark-sample-v2"
corpus = load_dataset(
    dataset_path,
    data_files="corpus.jsonl",
    split="train",
    # token=hf_token
).to_pandas()
queries = load_dataset(
    dataset_path,
    data_files="queries.jsonl",
    split="train",
    # token=hf_token
).to_pandas()


corpus["n_tokens"] = corpus["text"].apply(lambda x: len(tokenizer.encode(x)))
queries["n_tokens"] = queries["text"].apply(lambda x: len(tokenizer.encode(x)))


print("Corpus token count:", corpus["n_tokens"].sum())
print("Queries token count:", queries["n_tokens"].sum())

total_tokens = corpus["n_tokens"].sum() + queries["n_tokens"].sum()
print("Total tokens:", total_tokens)


Corpus token count: 29658310
Queries token count: 24955
Total tokens: 29683265


In [ ]:
# for NASA IR benchmark 
from datasets import load_dataset

dataset_path = "nasa-impact/nasa-smd-IR-benchmark"
corpus = load_dataset(
    dataset_path,
    data_files="corpus.jsonl",
    split="train",
    # token=hf_token
).to_pandas()
queries = load_dataset(
    dataset_path,
    data_files="queries.jsonl",
    split="train",
    # token=hf_token
).to_pandas()


corpus["n_tokens"] = corpus["text"].apply(lambda x: len(tokenizer.encode(x)))
queries["n_tokens"] = queries["text"].apply(lambda x: len(tokenizer.encode(x)))


print("Corpus token count:", corpus["n_tokens"].sum())
print("Queries token count:", queries["n_tokens"].sum())

total_tokens = corpus["n_tokens"].sum() + queries["n_tokens"].sum()
print("Total tokens:", total_tokens)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Corpus token count: 45133567
Queries token count: 7138
Total tokens: 45140705


In [18]:
# First, ensure you have the necessary libraries installed.
# You can run this cell to install them if you haven't already.
try:
    import datasets
    import tiktoken
except ImportError:
    print("Installing required libraries: datasets and tiktoken")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "tiktoken"])
    import datasets
    import tiktoken

from datasets import load_dataset
import pandas as pd

def count_tokens_for_dataset(dataset_name, tokenizer):
    """
    Loads a dataset, counts the tokens in its 'corpus' and 'queries' subsets,
    and returns the counts using the specified tiktoken tokenizer.

    Args:
        dataset_name (str): The name of the Hugging Face dataset.
        tokenizer: The tiktoken tokenizer instance to use for counting tokens.

    Returns:
        tuple: A tuple containing (corpus_tokens, queries_tokens).
               Returns (0, 0) if a subset is not found or on error.
    """
    corpus_tokens = 0
    queries_tokens = 0

    # --- Count tokens in the 'corpus' subset ---
    try:
        print(f"Loading corpus for {dataset_name}...")
        # Load the 'corpus' configuration and the 'train' split
        corpus_dataset = load_dataset(dataset_name, name='corpus', split='train', trust_remote_code=True)
        
        # Determine the correct text column for the corpus
        corpus_text_column = ''
        if 'text' in corpus_dataset.column_names:
            corpus_text_column = 'text'
        elif 'passage' in corpus_dataset.column_names:
            corpus_text_column = 'passage'
        else:
            print(f"Warning: Could not find a standard text column ('text', 'passage') in the corpus for {dataset_name}. Skipping corpus.")

        if corpus_text_column:
            print(f"Tokenizing corpus for {dataset_name} using column '{corpus_text_column}'...")
            # Batch processing for efficiency
            def tokenize_corpus(batch):
                nonlocal corpus_tokens
                # Filter out None values to prevent errors
                texts = [str(t) for t in batch[corpus_text_column] if t is not None]
                if texts:
                    # Use tiktoken's encode_batch for efficiency
                    encoded_texts = tokenizer.encode_batch(texts)
                    corpus_tokens += sum(len(ids) for ids in encoded_texts)
            
            corpus_dataset.map(tokenize_corpus, batched=True, batch_size=1000)

    except Exception as e:
        print(f"Could not load or process corpus for {dataset_name}. Error: {e}")

    # --- Count tokens in the 'queries' subset ---
    try:
        print(f"Loading queries for {dataset_name}...")
        # Load the 'queries' configuration and the 'train' split
        queries_dataset = load_dataset(dataset_name, name='queries', split='train', trust_remote_code=True)
        
        # Determine the correct text column for queries
        queries_text_column = ''
        if 'text' in queries_dataset.column_names:
            queries_text_column = 'text'
        elif 'query' in queries_dataset.column_names:
            queries_text_column = 'query'
        else:
            print(f"Warning: Could not find a standard text column ('text', 'query') in the queries for {dataset_name}. Skipping queries.")

        if queries_text_column:
            print(f"Tokenizing queries for {dataset_name} using column '{queries_text_column}'...")
            # Batch processing for efficiency
            def tokenize_queries(batch):
                nonlocal queries_tokens
                # Filter out None values to prevent errors
                texts = [str(t) for t in batch[queries_text_column] if t is not None]
                if texts:
                    # Use tiktoken's encode_batch for efficiency
                    encoded_texts = tokenizer.encode_batch(texts)
                    queries_tokens += sum(len(ids) for ids in encoded_texts)

            queries_dataset.map(tokenize_queries, batched=True, batch_size=1000)

    except Exception as e:
        print(f"Could not load or process queries for {dataset_name}. Error: {e}")

    return corpus_tokens, queries_tokens

def main():
    """
    Main function to iterate through datasets, count tokens, and display results.
    """
    # List of NanoBEIR datasets
    dataset_names = [
        "zeta-alpha-ai/NanoClimateFEVER",
        "zeta-alpha-ai/NanoDBPedia",
        "zeta-alpha-ai/NanoFEVER",
        "zeta-alpha-ai/NanoFiQA2018",
        "zeta-alpha-ai/NanoHotpotQA",
        "zeta-alpha-ai/NanoMSMARCO",
        "zeta-alpha-ai/NanoNFCorpus",
        "zeta-alpha-ai/NanoNQ",
        "zeta-alpha-ai/NanoQuoraRetrieval",
        "zeta-alpha-ai/NanoSCIDOCS",
        "zeta-alpha-ai/NanoArguAna",
        "zeta-alpha-ai/NanoSciFact",
        "zeta-alpha-ai/NanoTouche2020",
    ]

    # Using the tiktoken tokenizer as requested
    print("Initializing tokenizer...")
    tokenizer = tiktoken.get_encoding("cl100k_base")

    results = []
    total_corpus_tokens = 0
    total_queries_tokens = 0

    for name in dataset_names:
        print("-" * 50)
        print(f"Processing dataset: {name}")
        corpus_tokens, queries_tokens = count_tokens_for_dataset(name, tokenizer)
        
        results.append({
            "Dataset": name,
            "Corpus Tokens": corpus_tokens,
            "Queries Tokens": queries_tokens,
            "Total Tokens": corpus_tokens + queries_tokens
        })
        
        total_corpus_tokens += corpus_tokens
        total_queries_tokens += queries_tokens
        print(f"Finished processing {name}.")
        print(f"  Corpus Tokens: {corpus_tokens:,}")
        print(f"  Queries Tokens: {queries_tokens:,}")
        print(f"  Sub-Total: {(corpus_tokens + queries_tokens):,}")


    print("\n" + "=" * 70)
    print(" " * 25 + "Final Token Counts")
    print("=" * 70)

    # Display results in a formatted table using pandas
    df = pd.DataFrame(results)
    print(df.to_string(index=False, formatters={'Corpus Tokens': '{:,}'.format, 'Queries Tokens': '{:,}'.format, 'Total Tokens': '{:,}'.format}))

    print("-" * 70)
    grand_total = total_corpus_tokens + total_queries_tokens
    print(f"Total Corpus Tokens (All Datasets):  {total_corpus_tokens:,}")
    print(f"Total Queries Tokens (All Datasets): {total_queries_tokens:,}")
    print(f"Grand Total (All Datasets):          {grand_total:,}")
    print("=" * 70)

if __name__ == "__main__":
    main()


Initializing tokenizer...
--------------------------------------------------
Processing dataset: zeta-alpha-ai/NanoClimateFEVER
Loading corpus for zeta-alpha-ai/NanoClimateFEVER...
Tokenizing corpus for zeta-alpha-ai/NanoClimateFEVER using column 'text'...


Map:   0%|          | 0/3408 [00:00<?, ? examples/s]

Loading queries for zeta-alpha-ai/NanoClimateFEVER...
Tokenizing queries for zeta-alpha-ai/NanoClimateFEVER using column 'text'...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Finished processing zeta-alpha-ai/NanoClimateFEVER.
  Corpus Tokens: 1,119,546
  Queries Tokens: 1,329
  Sub-Total: 1,120,875
--------------------------------------------------
Processing dataset: zeta-alpha-ai/NanoDBPedia
Loading corpus for zeta-alpha-ai/NanoDBPedia...
Tokenizing corpus for zeta-alpha-ai/NanoDBPedia using column 'text'...


...

Finished processing zeta-alpha-ai/NanoSciFact.
  Corpus Tokens: 895,976
  Queries Tokens: 985
  Sub-Total: 896,961
--------------------------------------------------
Processing dataset: zeta-alpha-ai/NanoTouche2020
Loading corpus for zeta-alpha-ai/NanoTouche2020...
Tokenizing corpus for zeta-alpha-ai/NanoTouche2020 using column 'text'...


Map:   0%|          | 0/5745 [00:00<?, ? examples/s]

Loading queries for zeta-alpha-ai/NanoTouche2020...
Tokenizing queries for zeta-alpha-ai/NanoTouche2020 using column 'text'...


Map:   0%|          | 0/49 [00:00<?, ? examples/s]

Finished processing zeta-alpha-ai/NanoTouche2020.
  Corpus Tokens: 2,574,622
  Queries Tokens: 386
  Sub-Total: 2,575,008

                         Final Token Counts
                         Dataset Corpus Tokens Queries Tokens Total Tokens
  zeta-alpha-ai/NanoClimateFEVER     1,119,546          1,329    1,120,875
       zeta-alpha-ai/NanoDBPedia       478,066            360      478,426
         zeta-alpha-ai/NanoFEVER     1,344,835            569    1,345,404
      zeta-alpha-ai/NanoFiQA2018       912,956            608      913,564
      zeta-alpha-ai/NanoHotpotQA       418,852            981      419,833
       zeta-alpha-ai/NanoMSMARCO       364,528            331      364,859
      zeta-alpha-ai/NanoNFCorpus       965,977            267      966,244
            zeta-alpha-ai/NanoNQ       583,886            504      584,390
zeta-alpha-ai/NanoQuoraRetrieval        62,686            559       63,245
       zeta-alpha-ai/NanoSCIDOCS       381,411            654      382,065
       z

In [19]:
# for BEIR
# First, ensure you have the necessary libraries installed.
# You can run this cell to install them if you haven't already.
try:
    import tiktoken
    import pandas as pd
except ImportError:
    print("Installing required libraries: tiktoken and pandas")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken", "pandas"])
    import tiktoken
    import pandas as pd

import os
import json

def count_tokens_for_local_beir_dataset(dataset_path, tokenizer):
    """
    Loads a local BEIR dataset, counts the tokens in its corpus and queries,
    and returns the counts.

    Args:
        dataset_path (str): The path to the local BEIR dataset directory.
        tokenizer: The tiktoken tokenizer instance.

    Returns:
        tuple: A tuple containing (corpus_tokens, queries_tokens).
    """
    corpus_tokens = 0
    queries_tokens = 0

    # --- Process corpus file ---
    corpus_file_path = os.path.join(dataset_path, "corpus.jsonl")
    try:
        with open(corpus_file_path, 'r', encoding='utf-8') as f:
            corpus_texts = []
            for line in f:
                data = json.loads(line)
                # Combine title and text for a more comprehensive token count
                text = data.get('title', '') + " " + data.get('text', '')
                corpus_texts.append(text.strip())
            
            if corpus_texts:
                encoded_texts = tokenizer.encode_batch(corpus_texts)
                corpus_tokens = sum(len(ids) for ids in encoded_texts)

    except FileNotFoundError:
        print(f"Warning: corpus.jsonl not found in {dataset_path}")
    except Exception as e:
        print(f"Error processing corpus for {dataset_path}: {e}")

    # --- Process queries file ---
    queries_file_path = os.path.join(dataset_path, "queries.jsonl")
    try:
        with open(queries_file_path, 'r', encoding='utf-8') as f:
            queries_texts = [json.loads(line).get('text', '') for line in f]
            
            if queries_texts:
                encoded_texts = tokenizer.encode_batch(queries_texts)
                queries_tokens = sum(len(ids) for ids in encoded_texts)
                
    except FileNotFoundError:
        print(f"Warning: queries.jsonl not found in {dataset_path}")
    except Exception as e:
        print(f"Error processing queries for {dataset_path}: {e}")

    return corpus_tokens, queries_tokens


def main():
    """
    Main function to iterate through local BEIR datasets, count tokens, and display results.
    """
    # Using the tiktoken tokenizer as requested
    print("Initializing tokenizer...")
    tokenizer = tiktoken.get_encoding("cl100k_base")

    all_results = []
    grand_total_corpus = 0
    grand_total_queries = 0

    # --- Process local BEIR Datasets ---
    print("\n" + "=" * 70)
    print(" " * 22 + "Processing Local BEIR Datasets")
    print("=" * 70)
    beir_datasets = [
        "trec-covid", "nfcorpus", "nq", "hotpotqa", "fiqa", "arguana",
        "webis-touche2020", "dbpedia-entity", "scidocs", "fever",
        "climate-fever", "scifact"
    ]
    # IMPORTANT: Update this path if your local BEIR datasets directory changes
    base_path = "/rhome/sawale/indus_traning/sentense_transformers/eval/datasets"

    for name in beir_datasets:
        print("-" * 50)
        path = os.path.join(base_path, name)
        print(f"Processing local dataset: {name} from {path}")
        if not os.path.isdir(path):
            print(f"Warning: Directory not found for {name}, skipping.")
            continue
        
        corpus_tokens, queries_tokens = count_tokens_for_local_beir_dataset(path, tokenizer)
        all_results.append({
            "Dataset": name,
            "Corpus Tokens": corpus_tokens, 
            "Queries Tokens": queries_tokens,
            "Total Tokens": corpus_tokens + queries_tokens
        })
        grand_total_corpus += corpus_tokens
        grand_total_queries += queries_tokens
        print(f"Finished processing {name}. Tokens (Corpus: {corpus_tokens:,}, Queries: {queries_tokens:,})")

    # --- Display Final Results ---
    print("\n" + "=" * 80)
    print(" " * 30 + "Final Token Counts")
    print("=" * 80)

    df = pd.DataFrame(all_results)
    print(df.to_string(index=False, formatters={
        'Corpus Tokens': '{:,}'.format, 
        'Queries Tokens': '{:,}'.format, 
        'Total Tokens': '{:,}'.format
    }))

    print("-" * 80)
    grand_total = grand_total_corpus + grand_total_queries
    print(f"Total Corpus Tokens (All Datasets):  {grand_total_corpus:,}")
    print(f"Total Queries Tokens (All Datasets): {grand_total_queries:,}")
    print(f"Grand Total (All Datasets):          {grand_total:,}")
    print("=" * 80)

if __name__ == "__main__":
    main()


Initializing tokenizer...

                      Processing Local BEIR Datasets
--------------------------------------------------
Processing local dataset: trec-covid from /rhome/sawale/indus_traning/sentense_transformers/eval/datasets/trec-covid
Finished processing trec-covid. Tokens (Corpus: 40,206,651, Queries: 728)
--------------------------------------------------
Processing local dataset: nfcorpus from /rhome/sawale/indus_traning/sentense_transformers/eval/datasets/nfcorpus
Finished processing nfcorpus. Tokens (Corpus: 1,243,273, Queries: 17,207)
--------------------------------------------------
Processing local dataset: nq from /rhome/sawale/indus_traning/sentense_transformers/eval/datasets/nq
Finished processing nq. Tokens (Corpus: 290,409,097, Queries: 35,559)
--------------------------------------------------
Processing local dataset: hotpotqa from /rhome/sawale/indus_traning/sentense_transformers/eval/datasets/hotpotqa
Finished processing hotpotqa. Tokens (Corpus: 368,010,

In [21]:
all_results

NameError: name 'all_results' is not defined

In [2]:

from sentence_transformers.evaluation import NanoBEIREvaluator, dataset_name_to_id

ImportError: cannot import name 'dataset_name_to_id' from 'sentence_transformers.evaluation' (/rhome/sawale/indus_traning/sentense_transformers/stenv/lib/python3.11/site-packages/sentence_transformers/evaluation/__init__.py)